# AutoSignal Experiment Workbench

This is the fast research interface for the same pipeline used by the Streamlit
dashboard. It deliberately keeps the workflow explicit:

1. load one telemetry slice and one frozen agent configuration;
2. run every current method through the shared engine;
3. compare method performance and controls;
4. inspect one selected representation's signal regime;
5. optionally run a prespecified batch.

**Protocol:** feature/relation search is exploratory. Freeze a candidate before
using an untouched holdout, and do not use signal-regime results to repeatedly
modify the same labeled slice.

## 1. Imports and repository paths

The notebook imports `Notebooks/autosignal_engine.py`, so changes to the engine
are immediately available here and in Streamlit.

In [1]:
from __future__ import annotations

import copy
import gzip
import hashlib
import json
import os
import sys
import time
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "Notebooks" / "autosignal_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the AutoSignal repository.")


REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "Notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from autosignal_engine import (
    dataframe_profile,
    load_df_slice,
    run_autosignal,
    validate_config,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 80)
print("Repository:", REPO_ROOT)

W20260730 15:50:21.984462 134491313655936 agent.cpp:608] sysfs nodes path '/sys/class/kfd/kfd/topology/nodes' does not exist
/work/home/mwasti/tuttInstitute/Steering-Telemetry-Triage-with-Self-Supervised-Graph-Geometry/GraphTriage/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: /work/home/mwasti/tuttInstitute/Steering-Telemetry-Triage-with-Self-Supervised-Graph-Geometry


## 2. Edit experiment settings here

The defaults are intentionally small. Set `FEATURE_SET_NAMES` to a list when
you want to isolate particular agent hypotheses. Leave it as `None` to run all
feature sets in the JSON configuration.

Environment variables are supported for automated smoke tests; ordinary
notebook use only requires editing this cell.

In [2]:
DATASET_PATH = REPO_ROOT / "Data" / "train-process_uber_summary.parquet"
CONFIG_PATH = NOTEBOOK_DIR / "sample_agent_config.json"
CONFIG_INDEX = 0  # ACME object in the bundled multi-schema catalog

ROWS = int(os.getenv("AUTOSIGNAL_NOTEBOOK_ROWS", "1000"))
SLICE_KIND = "tail"  # "tail", "head", or "random"
SLICE_RANDOM_STATE = 42

FEATURE_SET_NAMES = None
# Example:
# FEATURE_SET_NAMES = ["Process Resource Consumption Profile"]

K = 15
SEEDS = (42,)
GRAPH_EPOCHS = int(os.getenv("AUTOSIGNAL_NOTEBOOK_EPOCHS", "2"))
SIGNAL_PERMUTATIONS = int(
    os.getenv("AUTOSIGNAL_NOTEBOOK_PERMUTATIONS", "99")
)

USE_CACHE = True
FORCE_RERUN = False
RUN_PIPELINE = os.getenv("AUTOSIGNAL_NOTEBOOK_RUN", "1") == "1"
CACHE_DIR = REPO_ROOT / "artifacts" / "autosignal_workbench_cache"

print(
    {
        "rows": ROWS,
        "slice": SLICE_KIND,
        "k": K,
        "seeds": SEEDS,
        "graph_epochs": GRAPH_EPOCHS,
        "signal_permutations": SIGNAL_PERMUTATIONS,
    }
)

{'rows': 1000, 'slice': 'tail', 'k': 15, 'seeds': (42,), 'graph_epochs': 2, 'signal_permutations': 99}


## 3. Load the slice and frozen agent hypotheses

Labels are retained for downstream development evaluation, but the engine does
not use them to construct features, train representations, score anomalies, or
select a signal regime.

In [ ]:
def load_slice(
    path: Path,
    rows: int,
    kind: str = "tail",
    random_state: int = 42,
) -> pd.DataFrame:
    if kind == "tail":
        return load_df_slice(path, rows)

    full = load_df_slice(path)
    if rows <= 0 or rows >= len(full):
        return full.reset_index(drop=True)
    if kind == "head":
        return full.head(rows).reset_index(drop=True).copy()
    if kind == "random":
        return full.sample(rows, random_state=random_state).reset_index(drop=True)
    raise ValueError("SLICE_KIND must be 'tail', 'head', or 'random'.")


dataset = load_slice(
    DATASET_PATH,
    ROWS,
    kind=SLICE_KIND,
    random_state=SLICE_RANDOM_STATE,
)
config_document = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
agent_config = copy.deepcopy(
    config_document[CONFIG_INDEX]
    if isinstance(config_document, list)
    else config_document
)

if FEATURE_SET_NAMES is not None:
    requested = set(FEATURE_SET_NAMES)
    agent_config["feature_sets"] = [
        item
        for item in agent_config["feature_sets"]
        if item["name"] in requested
    ]
    found = {item["name"] for item in agent_config["feature_sets"]}
    missing = requested - found
    if missing:
        raise ValueError(f"Unknown feature-set names: {sorted(missing)}")
    retained_columns = {
        column
        for feature_set in agent_config["feature_sets"]
        for column in feature_set["columns"]
    }
    agent_config["selected_columns"] = [
        item
        for item in agent_config["selected_columns"]
        if item["column"] in retained_columns
    ]

validated_config = validate_config(agent_config, dataset)
profile = dataframe_profile(dataset)

print(f"Loaded {len(dataset):,} rows and {len(dataset.columns):,} columns")
display(
    pd.DataFrame(agent_config["feature_sets"])[
        ["name", "columns", "rationale"]
    ]
)
display(
    pd.DataFrame(agent_config["graph_relations"])[
        [
            "relation_name",
            "source_node_type",
            "source_column",
            "target_node_type",
            "target_column",
        ]
    ]
)

## 4. Run or load the complete pipeline

The cache key includes the dataset file signature, slice definition,
configuration, scoring settings, and seeds. Changing any of those produces a
new result. Delete the cache or set `FORCE_RERUN=True` after changing engine
implementation details.

In [ ]:
def cache_identity(
    dataset_path: Path,
    config: dict,
    *,
    rows: int,
    slice_kind: str,
    slice_random_state: int,
    k: int,
    seeds: tuple[int, ...],
    graph_epochs: int,
    signal_permutations: int,
) -> str:
    stat = dataset_path.stat()
    identity = {
        "engine_result_schema_version": "2.0",
        "engine_sha256": hashlib.sha256(
            (NOTEBOOK_DIR / "autosignal_engine.py").read_bytes()
        ).hexdigest(),
        "analysis_sha256": hashlib.sha256(
            (NOTEBOOK_DIR / "autosignal_analysis.py").read_bytes()
        ).hexdigest(),
        "dataset": str(dataset_path.resolve()),
        "dataset_size": stat.st_size,
        "dataset_mtime_ns": stat.st_mtime_ns,
        "rows": rows,
        "slice_kind": slice_kind,
        "slice_random_state": slice_random_state,
        "config": config,
        "k": k,
        "seeds": list(seeds),
        "graph_epochs": graph_epochs,
        "signal_permutations": signal_permutations,
    }
    encoded = json.dumps(identity, sort_keys=True, default=str).encode()
    return hashlib.sha256(encoded).hexdigest()[:16]


def run_cached(
    df: pd.DataFrame,
    config: dict,
    *,
    cache_name: str,
    k: int,
    seeds: tuple[int, ...],
    graph_epochs: int,
    signal_permutations: int,
    use_cache: bool = True,
    force: bool = False,
) -> dict:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = CACHE_DIR / f"{cache_name}.json.gz"
    if use_cache and cache_path.exists() and not force:
        print("Loading cached result:", cache_path.name)
        with gzip.open(cache_path, "rt", encoding="utf-8") as handle:
            return json.load(handle)

    last_report = {"time": 0.0, "message": None}

    def progress(fraction: float, message: str) -> None:
        now = time.monotonic()
        if (
            message != last_report["message"]
            or now - last_report["time"] >= 5
            or fraction >= 1
        ):
            print(f"{fraction:6.1%}  {message}")
            last_report.update(time=now, message=message)

    result = run_autosignal(
        df,
        config,
        k=k,
        seeds=seeds,
        graph_epochs=graph_epochs,
        signal_permutations=signal_permutations,
        progress_callback=progress,
    )
    if use_cache:
        with gzip.open(cache_path, "wt", encoding="utf-8") as handle:
            json.dump(result, handle, allow_nan=False)
        print("Saved cache:", cache_path.name)
    return result


cache_key = cache_identity(
    DATASET_PATH,
    agent_config,
    rows=ROWS,
    slice_kind=SLICE_KIND,
    slice_random_state=SLICE_RANDOM_STATE,
    k=K,
    seeds=SEEDS,
    graph_epochs=GRAPH_EPOCHS,
    signal_permutations=SIGNAL_PERMUTATIONS,
)

result = None
if RUN_PIPELINE:
    result = run_cached(
        dataset,
        agent_config,
        cache_name=cache_key,
        k=K,
        seeds=SEEDS,
        graph_epochs=GRAPH_EPOCHS,
        signal_permutations=SIGNAL_PERMUTATIONS,
        use_cache=USE_CACHE,
        force=FORCE_RERUN,
    )
    print("Run status:", result["status"])
else:
    print("RUN_PIPELINE=False; configuration loaded without running methods.")

## 5. Normalize the result payload into tables

`method_key` is the join key connecting method metrics, per-session scores,
embeddings, and signal results.

In [ ]:
if result is None:
    raise RuntimeError("Set RUN_PIPELINE=True and run the previous cell first.")


def method_key_from_row(row: pd.Series) -> str:
    if pd.notna(row.get("method_key")):
        return str(row["method_key"])
    key = str(row["method"])
    hypothesis = row.get("hypothesis")
    seed = row.get("seed")
    if pd.notna(hypothesis) and hypothesis is not None:
        key += f"::{hypothesis}"
    if pd.notna(seed) and seed is not None:
        key += f"::seed={int(seed)}"
    return key


methods = pd.DataFrame(result["method_results"])
methods["method_key"] = methods.apply(method_key_from_row, axis=1)
completed = methods[methods["status"].eq("completed")].copy()
failed = methods[~methods["status"].eq("completed")].copy()
scores = pd.DataFrame(result["session_scores"])
embeddings = pd.DataFrame(result["embeddings"])
sessions = pd.DataFrame(result["sessions"])
signal_results = pd.DataFrame(result.get("signal_results", []))
signal_scores = pd.DataFrame(result.get("signal_scores", []))
representation_results = pd.DataFrame(result.get("representation_results", []))
representation_stability = pd.DataFrame(result.get("representation_stability", []))
scorer_diagnostics = pd.DataFrame(result.get("scorer_diagnostics", []))
score_components = pd.DataFrame(result.get("score_components", []))

print(
    f"{len(completed)} completed methods · {len(failed)} non-completed outcomes · "
    f"{len(sessions)} sessions"
)
display(pd.DataFrame([result["run_manifest"]]))
display(pd.DataFrame([result["session_manifest"]]))
if not representation_results.empty:
    display(representation_results.drop(columns=["dimension_names"], errors="ignore"))
if not failed.empty:
    display(failed[["method", "hypothesis", "seed", "status", "error"]])

## 6. Method leaderboard

This table is development evidence, not final confirmation. Average precision
and analyst-budget metrics are only populated when labels exist.

In [ ]:
leaderboard_columns = [
    "representation",
    "scorer",
    "method",
    "hypothesis",
    "representation_seed",
    "scorer_seed",
    "average_precision",
    "reviews_to_first_malicious",
    "found_at_25",
    "recall_at_25",
    "found_at_100",
    "recall_at_100",
    "method_key",
]
leaderboard_columns = [
    column for column in leaderboard_columns if column in completed
]
leaderboard = completed[leaderboard_columns].sort_values(
    ["average_precision", "recall_at_100"],
    ascending=[False, False],
    na_position="last",
)
display(leaderboard.reset_index(drop=True))

### GraphSAGE trained-versus-random control

A positive delta means training improved over the same random encoder
architecture for the same feature hypothesis and seed. Treat tiny deltas as
negligible until uncertainty and holdout behavior support them.

In [ ]:
graphsage = completed[
    completed["method"].isin(["graphsage_random", "graphsage_trained"])
].copy()

control_metrics = [
    "average_precision",
    "recall_at_25",
    "recall_at_100",
    "reviews_to_first_malicious",
]
control_metrics = [column for column in control_metrics if column in graphsage]

if graphsage.empty:
    print("No completed GraphSAGE controls.")
else:
    comparison = graphsage.pivot_table(
        index=["hypothesis", "seed"],
        columns="method",
        values=control_metrics,
        aggfunc="first",
    )
    comparison.columns = [
        f"{metric}__{method}" for metric, method in comparison.columns
    ]
    comparison = comparison.reset_index()
    for metric in control_metrics:
        trained = f"{metric}__graphsage_trained"
        random = f"{metric}__graphsage_random"
        if trained in comparison and random in comparison:
            comparison[f"delta__{metric}"] = (
                comparison[trained] - comparison[random]
            )
    display(comparison)

## 7. Graph construction diagnostics

Check relation coverage before interpreting a graph method. Sparse or
near-universal relations can make trained and random graph encoders behave
similarly.

In [ ]:
graph_manifest = result.get("graph_manifest") or {}
display(
    pd.DataFrame(
        [
            {
                "primary_node_type": graph_manifest.get("primary_node_type"),
                "primary_id_column": graph_manifest.get("primary_id_column"),
                "forward_edges": graph_manifest.get("total_forward_edges"),
                "observed_primary_nodes": graph_manifest.get(
                    "observed_primary_nodes"
                ),
                "external_primary_nodes": graph_manifest.get(
                    "external_primary_nodes"
                ),
            }
        ]
    )
)
display(pd.DataFrame(graph_manifest.get("relations", [])))
display(
    pd.DataFrame(
        [
            {"node_type": key, "nodes": value}
            for key, value in graph_manifest.get("node_counts", {}).items()
        ]
    )
)

## 8. Signal-regime leaderboard

Diagnosis is label-blind and runs on each full representation, not UMAP.
Browsing many representations is exploratory because each family-wise p-value
only corrects the three regime hypotheses within one representation.

In [ ]:
def flatten_signal_result(record: dict) -> dict:
    regime = record.get("regime")
    winner = next(
        (
            item
            for item in record.get("evidence", [])
            if item.get("regime") == regime
        ),
        {},
    )
    evaluation = record.get("evaluation") or {}
    intrinsic = evaluation.get("intrinsic") or {}
    matched = evaluation.get("matched") or {}
    intrinsic_ap = intrinsic.get("average_precision")
    matched_ap = matched.get("average_precision")
    return {
        "method_key": record.get("method_key"),
        "method": record.get("method"),
        "hypothesis": record.get("hypothesis"),
        "seed": record.get("seed"),
        "status": record.get("status"),
        "regime": regime,
        "effect_z": winner.get("effect_z"),
        "familywise_p": winner.get("familywise_p"),
        "passing_regimes": sum(
            bool(item.get("passes")) for item in record.get("evidence", [])
        ),
        "intrinsic_ap": intrinsic_ap,
        "matched_ap": matched_ap,
        "matched_ap_delta": (
            matched_ap - intrinsic_ap
            if intrinsic_ap is not None and matched_ap is not None
            else None
        ),
    }


signal_summary = pd.DataFrame(
    [flatten_signal_result(record) for record in result.get("signal_results", [])]
)
if signal_summary.empty:
    print("No signal results.")
else:
    display(
        signal_summary.sort_values(
            ["familywise_p", "effect_z"],
            ascending=[True, False],
            na_position="last",
        ).reset_index(drop=True)
    )

## 9. Inspect one representation interactively

Choose a `method_key` from the leaderboard. Color is based on the selected
signal channel's within-run percentile, while hover values preserve raw units.
Labels are omitted from the plot by default.

In [ ]:
available_signal_keys = (
    signal_results.loc[
        signal_results["status"].eq("completed"), "method_key"
    ].tolist()
    if not signal_results.empty
    else []
)

if not available_signal_keys:
    raise RuntimeError("No completed signal representation is available.")

SELECTED_METHOD_KEY = available_signal_keys[0]
CHANNEL = "matched_score"
# Channels: intrinsic, neighbor_support, second_hop_support, local_contrast,
# local_contrast_magnitude, pocket, supported_candidate, matched_score

selected_signal = signal_scores[
    signal_scores["method_key"].eq(SELECTED_METHOD_KEY)
].copy()
selected_embedding = embeddings[
    embeddings["method_key"].eq(SELECTED_METHOD_KEY)
][["session_id", "x", "y"]]

plot_data = (
    selected_signal.merge(
        selected_embedding,
        on="session_id",
        validate="one_to_one",
    )
    .merge(
        sessions.drop(columns=["label"], errors="ignore"),
        on="session_id",
        validate="one_to_one",
    )
)
plot_data["channel_percentile"] = plot_data[CHANNEL].rank(
    pct=True,
    method="average",
)

signal_chart = (
    alt.Chart(plot_data)
    .mark_circle(opacity=0.78, size=55)
    .encode(
        x=alt.X("x:Q", title="UMAP 1"),
        y=alt.Y("y:Q", title="UMAP 2"),
        color=alt.Color(
            "channel_percentile:Q",
            title=f"{CHANNEL} percentile",
            scale=alt.Scale(scheme="turbo"),
        ),
        tooltip=[
            alt.Tooltip("session_id:O"),
            alt.Tooltip(f"{CHANNEL}:Q", format=".5f"),
            alt.Tooltip("intrinsic:Q", format=".5f"),
            alt.Tooltip("neighbor_support:Q", format=".5f"),
            alt.Tooltip("second_hop_support:Q", format=".5f"),
            alt.Tooltip("local_contrast:Q", format=".5f"),
            alt.Tooltip("pocket:Q", format=".5f"),
            alt.Tooltip("matched_score:Q", format=".5f"),
            alt.Tooltip("intrinsic_rank:Q"),
            alt.Tooltip("matched_rank:Q"),
            alt.Tooltip("rank_gain:Q"),
            alt.Tooltip("rows:Q"),
            alt.Tooltip("start_time:N"),
            alt.Tooltip("end_time:N"),
        ],
    )
    .properties(
        width=850,
        height=520,
        title=f"{SELECTED_METHOD_KEY} · {CHANNEL}",
    )
    .interactive()
)
signal_chart

### Evidence and analyst queue for the selected representation

Reveal labels only when evaluating a frozen development result. Keep them out
of exploratory visual interpretation whenever possible.

In [ ]:
REVEAL_DEVELOPMENT_LABELS = False
TOP_N = 25

selected_record = signal_results[
    signal_results["method_key"].eq(SELECTED_METHOD_KEY)
].iloc[0]
print("Selected regime:", selected_record["regime"])
display(pd.DataFrame(selected_record["evidence"]))

queue = (
    selected_signal.merge(
        sessions,
        on=["session_id", "label"],
        how="left",
        validate="one_to_one",
    )
    .sort_values("matched_rank")
    .reset_index(drop=True)
)
queue_columns = [
    "matched_rank",
    "session_id",
    "intrinsic",
    "matched_score",
    "intrinsic_rank",
    "rank_gain",
    "rows",
    "start_time",
    "end_time",
]
if REVEAL_DEVELOPMENT_LABELS:
    queue_columns.insert(6, "label")
display(queue[queue_columns].head(TOP_N))

evaluation = selected_record.get("evaluation") or {}
display(
    pd.DataFrame(
        [
            {"ranking": ranking, **metrics}
            for ranking, metrics in evaluation.items()
            if isinstance(metrics, dict)
        ]
    )
)

## 10. Paired bootstrap for one prespecified comparison

Use this only after choosing a challenger and baseline. It resamples aligned
sessions and reports uncertainty for the AP difference. It does **not** correct
for searching across many feature/method combinations.

In [ ]:
def paired_ap_bootstrap(
    payload: dict,
    challenger_key: str,
    baseline_key: str,
    *,
    repeats: int = 1000,
    random_state: int = 42,
) -> pd.Series:
    score_table = pd.DataFrame(payload["session_scores"])
    challenger = score_table[
        score_table["method_key"].eq(challenger_key)
    ][["session_id", "score", "label"]].rename(
        columns={"score": "challenger"}
    )
    baseline = score_table[
        score_table["method_key"].eq(baseline_key)
    ][["session_id", "score"]].rename(columns={"score": "baseline"})
    paired = challenger.merge(
        baseline,
        on="session_id",
        validate="one_to_one",
    )
    y = paired["label"].eq("malicious").to_numpy(dtype=int)
    if y.sum() == 0 or y.sum() == len(y):
        raise ValueError("Paired AP bootstrap requires both label classes.")

    observed = average_precision_score(y, paired["challenger"]) - (
        average_precision_score(y, paired["baseline"])
    )
    rng = np.random.default_rng(random_state)
    deltas = []
    for _ in range(repeats):
        indices = rng.integers(0, len(paired), len(paired))
        sampled_y = y[indices]
        if sampled_y.sum() == 0 or sampled_y.sum() == len(sampled_y):
            continue
        deltas.append(
            average_precision_score(
                sampled_y,
                paired["challenger"].to_numpy()[indices],
            )
            - average_precision_score(
                sampled_y,
                paired["baseline"].to_numpy()[indices],
            )
        )
    if not deltas:
        raise ValueError("No valid bootstrap resamples; the slice is too small.")
    return pd.Series(
        {
            "challenger": challenger_key,
            "baseline": baseline_key,
            "observed_ap_delta": observed,
            "ci_2.5%": np.quantile(deltas, 0.025),
            "ci_97.5%": np.quantile(deltas, 0.975),
            "valid_resamples": len(deltas),
        }
    )


# Example: compare trained GraphSAGE against its matched random control.
RUN_PAIRED_BOOTSTRAP = False
CHALLENGER_KEY = None
BASELINE_KEY = None

if RUN_PAIRED_BOOTSTRAP:
    if not CHALLENGER_KEY or not BASELINE_KEY:
        raise ValueError("Set CHALLENGER_KEY and BASELINE_KEY first.")
    display(
        paired_ap_bootstrap(
            result,
            CHALLENGER_KEY,
            BASELINE_KEY,
        ).to_frame("value")
    )
else:
    print("Set comparison keys and RUN_PAIRED_BOOTSTRAP=True when prespecified.")

## 11. Optional batch experiment scaffold

The engine already runs all configured feature hypotheses and all supplied
seeds. This loop adds multiple fixed data slices. Start with three meaningful
slices and avoid a huge search until the experiment contract is stable.

In [ ]:
RUN_BATCH = False
BATCH_SEEDS = (42, 43, 44, 45, 46)
BATCH_GRAPH_EPOCHS = GRAPH_EPOCHS
BATCH_SIGNAL_PERMUTATIONS = SIGNAL_PERMUTATIONS

SLICE_SPECS = [
    {"name": "recent_1000", "kind": "tail", "rows": 1000, "random_state": 42},
    # Add meaningfully different, prespecified slices:
    # {"name": "early_1000", "kind": "head", "rows": 1000, "random_state": 42},
    # {"name": "random_1000", "kind": "random", "rows": 1000, "random_state": 42},
]


def run_batch(
    dataset_path: Path,
    config: dict,
    slice_specs: list[dict],
) -> tuple[pd.DataFrame, dict[str, dict]]:
    batch_rows = []
    payloads = {}
    for spec in slice_specs:
        print("\n===", spec["name"], "===")
        batch_df = load_slice(
            dataset_path,
            spec["rows"],
            kind=spec["kind"],
            random_state=spec.get("random_state", 42),
        )
        batch_key = cache_identity(
            dataset_path,
            config,
            rows=spec["rows"],
            slice_kind=spec["kind"],
            slice_random_state=spec.get("random_state", 42),
            k=K,
            seeds=BATCH_SEEDS,
            graph_epochs=BATCH_GRAPH_EPOCHS,
            signal_permutations=BATCH_SIGNAL_PERMUTATIONS,
        )
        payload = run_cached(
            batch_df,
            config,
            cache_name=batch_key,
            k=K,
            seeds=BATCH_SEEDS,
            graph_epochs=BATCH_GRAPH_EPOCHS,
            signal_permutations=BATCH_SIGNAL_PERMUTATIONS,
            use_cache=USE_CACHE,
            force=FORCE_RERUN,
        )
        payloads[spec["name"]] = payload
        rows = pd.DataFrame(payload["method_results"])
        rows.insert(0, "slice", spec["name"])
        rows.insert(1, "slice_rows", len(batch_df))
        batch_rows.append(rows)
    return pd.concat(batch_rows, ignore_index=True), payloads


batch_results = None
batch_payloads = {}
if RUN_BATCH:
    batch_results, batch_payloads = run_batch(
        DATASET_PATH,
        agent_config,
        SLICE_SPECS,
    )
    display(
        batch_results.sort_values(
            ["slice", "average_precision"],
            ascending=[True, False],
            na_position="last",
        )
    )
else:
    print("RUN_BATCH=False; no batch jobs launched.")

## 12. Batch stability summary

This summary is intentionally descriptive. Formal claims should use
prespecified paired comparisons and an untouched confirmation split.

In [ ]:
if batch_results is None:
    print("Run the optional batch cell first.")
else:
    completed_batch = batch_results[
        batch_results["status"].eq("completed")
    ].copy()
    stability = (
        completed_batch.groupby(["method", "hypothesis"], dropna=False)
        .agg(
            runs=("average_precision", "size"),
            slices=("slice", "nunique"),
            mean_ap=("average_precision", "mean"),
            std_ap=("average_precision", "std"),
            mean_recall_at_100=("recall_at_100", "mean"),
            std_recall_at_100=("recall_at_100", "std"),
        )
        .reset_index()
        .sort_values("mean_ap", ascending=False, na_position="last")
    )
    display(stability)

    output_path = (
        REPO_ROOT / "artifacts" / "autosignal_batch_method_results.csv"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    batch_results.to_csv(output_path, index=False)
    print("Saved:", output_path)

## Iteration checklist

Before interpreting a result:

- Confirm the feature/schema agent received a versioned feature catalog with
  meanings, granularity, units, derivations, valid aggregations, availability,
  missing-value behavior, and leakage restrictions. A dataframe schema alone
  is not sufficient for defensible feature proposals.
- Confirm the slice has enough malicious sessions for the requested review
  budgets.
- Check relation coverage and method failures.
- Compare trained GraphSAGE with its same-seed random control.
- Prefer stable effects across seeds/slices over one high score.
- Treat cross-method and cross-feature browsing as exploratory.
- Freeze one candidate before signal diagnosis becomes confirmatory.
- Apply at most one matched amplification and reject it if analyst utility
  worsens.
- Confirm the final choice on untouched data.

To add a future method, implement it once in `autosignal_engine.py` and emit the
same aligned method, score, embedding, and signal artifacts. This notebook and
the Streamlit workflow will then evaluate it through the same contract.

In [ ]:
with open(CONFIG_PATH, "r") as f:
    x = json.loads(f.read())


x[-1]

In [41]:
json.dumps((json.loads(CONFIG_PATH.read_text(encoding="utf-8"))[0]))

'{"timestamp_col": "process_started", "existing_session_id_col": null, "session_group_cols": ["hostname", "user_name"], "graph_relations": [{"relation_name": "parent_of", "source_node_type": "process", "source_column": "parent_pid_hash", "target_node_type": "process", "target_column": "pid_hash", "directed": true, "meaning": "Establishes execution lineage from parent to child process."}, {"relation_name": "executes", "source_node_type": "process", "source_column": "pid_hash", "target_node_type": "executable", "target_column": "file_sha2", "directed": true, "meaning": "Links a process to its executable image hash."}, {"relation_name": "runs_on", "source_node_type": "process", "source_column": "pid_hash", "target_node_type": "host", "target_column": "hostname", "directed": true, "meaning": "Associates a process with the endpoint where it executed."}, {"relation_name": "owned_by", "source_node_type": "process", "source_column": "pid_hash", "target_node_type": "user", "target_column": "use